In [ ]:
#semi failed experement, might need some finetuning, but here we make a non-semantic-sensetive data augmentation on full conversation
#we take conversations and change a word here and there, delete a word swap a word and so on, might worth considatation
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split
import warnings
import random
warnings.filterwarnings("ignore")

/home/ronfr/.conda/envs/alephbert_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def word_tokenize(text):
    # Tokenize with is_split_into_words=False to get subwords
    encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    tokens = []
    prev_offset = None
    current_word = ""
    
    for token, (start, end) in zip(tokenizer.convert_ids_to_tokens(encoding['input_ids']), encoding['offset_mapping']):
        if prev_offset is not None and start != prev_offset:
            # New word detected
            tokens.append(current_word)
            current_word = ""
        current_word += text[start:end]
        prev_offset = end

    if current_word:
        tokens.append(current_word)

    return tokens


def augment_conversation(conv, synonyms_dict):
    augmented_conv = []

    for message in conv:
        op = random.choices(
            ["nothing", "synonym", "delete", "swap"],
            weights=[0.7, 0.1, 0.1, 0.1],
            k=1
        )[0]

        # Tokenize using SpaCy
        words = word_tokenize(message)

        if op == "nothing" or len(words) == 0:
            augmented_conv.append(message)
            continue

        if op == "synonym":
            candidates = [i for i, word in enumerate(words) if word in synonyms_dict]
            if candidates:
                idx = random.choice(candidates)
                synonym = random.choice(synonyms_dict[words[idx]])
                words[idx] = synonym

        elif op == "delete" and len(words) > 1:
            idx = random.randrange(len(words))
            del words[idx]

        elif op == "swap" and len(words) > 1:
            idx1, idx2 = random.sample(range(len(words)), 2)
            words[idx1], words[idx2] = words[idx2], words[idx1]

        # Join tokens back (just using spaces; punctuation is preserved by tokenizer)
        new_message = ' '.join(words)
        augmented_conv.append(new_message)

    return augmented_conv

synonyms_dict = {
    "שמח": ["מאושר", "עליז", "מרוצה"],
    "עצוב": ["מדוכא", "מדוכדך", "נוגה"],
    "עכשיו": ["כרגע", "בעת הזו", "כעת"],
    "מהר": ["בזריזות", "במהירות", "חיש מהר"],
    "יפה": ["נאה", "מרהיב", "מושך"],
    "רע": ["גרוע", "מבאס", "נורא"],
    "טוב": ["נחמד", "מעולה", "מצוין"],
    "גדול": ["ענק", "רחב", "עצום"],
    "קטן": ["זעיר", "מצומצם", "מינימלי"],
    "אני": ["אנוכי", "הנני"],
    "חבר": ["ידיד", "מכר", "עמית"],
    "עבודה": ["משרה", "תפקיד", "עיסוק"],
    "לבד": ["יחידי", "עצמאי", "לבדי"],
    "אהבה": ["חיבה", "רגש", "תשוקה"],
    "פחד": ["חשש", "יראה", "בהלה"],
    "שונא": ["מתעב", "מתנגד", "סולד"],
    "כועס": ["רגוז", "זועם", "נרגז"],
    "רעב": ["מזדקק לאוכל", "כפוי בטנו", "מת מרעב"],
    "עייף": ["מותש", "יגע", "תשוש"],
    "ישן": ["עתיק", "לא חדש", "מיושן"],
    "חדש": ["עדכני", "רענן", "טרי"],
    "מעניין": ["מסקרן", "מושך", "מרתק"],
    "משעמם": ["חדגוני", "מעייף", "חסר עניין"],
    "רגוע": ["שלו", "שקט", "נינוח"],
    "מה": ["איזה", "כיצד", "מדוע"],
    "למה": ["מדוע", "מה הסיבה", "בשביל מה"],
    "כן": ["בהחלט", "ודאי", "נכון"],
    "לא": ["שלילי", "בשום פנים", "אין מצב"],
    "בבקשה": ["אנא", "אם אפשר", "תעשה טובה"],
    "תודה": ["תודה רבה", "מוערך", "אני מודה"],
    "סליחה": ["מחילה", "אני מתנצל", "סלח לי"],
}

In [3]:
conv_info_path = 'trainDatasets/conv_info.csv'
conv_info_df = pd.read_csv(conv_info_path)
conv_info_df['engagement_id'] = conv_info_df['engagement_id'].astype(str)
conv_info_df = conv_info_df.rename(columns={'gsr': 'label'})
train_conv_df, test_conv_df = train_test_split(conv_info_df, test_size=0.2, stratify=conv_info_df['label'],random_state=42)

all_messages_path = 'trainDatasets/messages_anonymized.csv'

all_messages_df = pd.read_csv(all_messages_path)

all_messages_df['engagement_id'] = all_messages_df['engagement_id'].astype(str)
all_messages_df = all_messages_df[all_messages_df['anonymized'].notna()]
all_messages_df['name'] = all_messages_df['name'].fillna('-')
all_messages_df = all_messages_df[all_messages_df["seeker"]]

#split to train and test base on conversation split and concat messages
train_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(train_conv_df["engagement_id"])]
train_all_messages_df = train_all_messages_df.merge(train_conv_df , on="engagement_id")
train_all_messages_df = train_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

test_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(test_conv_df["engagement_id"])]
test_all_messages_df = test_all_messages_df.merge(test_conv_df , on="engagement_id")
test_all_messages_df = test_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

In [4]:
results = []

for _, row in train_all_messages_df.iterrows():
    row['augmented'] = 1
    results.append(row)
    for i in range(3):
        new_row = row.copy()
        new_row['anonymized'] = "[SEP]".join(augment_conversation(row['anonymized'].split("[SEP]"), synonyms_dict))
        new_row['augmented'] = 0
        results.append(new_row)

results_df = pd.DataFrame(results)
results_df.reset_index(inplace=True)

Token indices sequence length is longer than the specified maximum sequence length for this model (560 > 512). Running this sequence through the model will result in indexing errors


In [5]:
results_df.to_csv('trainDatasets/low_resrc_augmented_trainset_4_times_bigger.csv', index=False)